In [17]:
import os
for dirname, _, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(dirname, file))

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [18]:
# all imports 
import pandas as pd
from sklearn.pipeline import Pipeline

In [19]:
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
#train.head()

In [20]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [21]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [22]:
df = train.copy()
df['Titles'] = df['Name'].str.split(",").str[1]
df['Titles'] = df['Titles'].str.split(".").str[0].str.strip()
counts = df["Titles"].value_counts()
rare = counts[counts < 10].index.to_list()
rare

['Dr',
 'Rev',
 'Col',
 'Mlle',
 'Major',
 'Ms',
 'Mme',
 'Don',
 'Lady',
 'Sir',
 'Capt',
 'the Countess',
 'Jonkheer']

In [23]:
def feature_engineering(df):
    df = df.copy()
    df['Titles'] = df['Name'].str.split(",").str[1]
    df['Titles'] = df['Titles'].str.split(".").str[0].str.strip()
    
    counts = df["Titles"].value_counts()
    rare_titles = counts[counts < 10].index.to_list()
    
    df["Titles"] = df["Titles"].replace(rare_titles, "Rare")
    df['Titles'] = df["Titles"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })
    
    df["FamilySize"] = df["Parch"] + df["SibSp"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
    
    
    df["Deck"] = df["Cabin"].str[0]
    df["Deck"]= df["Deck"].fillna("Unknown")
    return df

In [24]:
train = feature_engineering(train)
test = feature_engineering(test)

In [25]:
train['Age'] = train.groupby('Titles')["Age"].transform(
    lambda x: x.fillna(x.median())
)
test["Age"] = test.groupby("Titles")["Age"].transform(
    lambda x: x.fillna(x.median())
)
train["Age"].fillna(train["Age"].median())
test["Age"].fillna(test["Age"].median())

0      34.5
1      47.0
2      62.0
3      27.0
4      22.0
       ... 
413    28.5
414    39.0
415    38.5
416    28.5
417     7.0
Name: Age, Length: 418, dtype: float64

In [26]:
train[:1]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Titles,FamilySize,IsAlone,FarePerPerson,Deck
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S,Mr,2,0,3.625,Unknown


In [27]:
X = train.drop(columns = [
    'PassengerId',
    'Name',
    'Ticket',
    'Cabin',
    'Survived'
])
X_test = test.drop(columns = [
    'PassengerId',
    'Name',
    'Ticket',
    'Cabin'
]) 
y = train["Survived"]
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Titles,FamilySize,IsAlone,FarePerPerson,Deck
0,3,male,22.0,1,0,7.2500,S,Mr,2,0,3.62500,Unknown
1,1,female,38.0,1,0,71.2833,C,Mrs,2,0,35.64165,C
2,3,female,26.0,0,0,7.9250,S,Miss,1,1,7.92500,Unknown
3,1,female,35.0,1,0,53.1000,S,Mrs,2,0,26.55000,C
4,3,male,35.0,0,0,8.0500,S,Mr,1,1,8.05000,Unknown


In [28]:
numerical = X.select_dtypes(include = 'number').columns
categorical = X.select_dtypes(exclude = 'number').columns

In [29]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder as ohe
from sklearn.compose import ColumnTransformer as ct

In [30]:
num_transformer = Pipeline(
    steps = [("imputer", SimpleImputer(strategy = 'median'))])
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("oneHot", ohe(handle_unknown="ignore"))
])

In [31]:
preprocessor = ct(transformers = [
    ('num', num_transformer, numerical),
    ('cat', cat_transformer, categorical)])


In [32]:
from sklearn.ensemble import RandomForestClassifier

In [33]:
model = RandomForestClassifier(
    n_estimators=500, 
    max_depth=7,
    min_samples_split=6,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [34]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)])

In [35]:
from sklearn.model_selection import train_test_split as tts
X_train, X_val, y_train, y_test = tts(X, y, 
                                     test_size=0.2,
                                     random_state=42,
                                     stratify=y)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone',
       'FarePerPerson'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('oneHot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Sex', 'Embarked', 'Titles', 'Deck'], dtype='object'))])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=7,
                                        min_samples_leaf=2, min_samples_split=6,
                                        n_estimators=500, n_jobs=-1,
                                        random_state=42))])

In [38]:
pipeline.fit(X, y)
predictions = pipeline.predict(X_test)

In [39]:
submission = pd.DataFrame({
    "PassengerId":test["PassengerId"],
    "Survived":predictions
})
submission.to_csv("submission.csv", 
                 index = False)
print(submission.head())

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
